In [ ]:
#confirm device type
import tensorflow as tf

print("TensorFlow version:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

TensorFlow version: 2.20.0
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [ ]:
#Mount google drive
from google.colab import drive
drive.mount('/content/drive')

#Check that Colab can see your ZIP file
import os

ZIP_PATH = "/content/drive/MyDrive/ML/Dataset.zip"

print("Dataset exists:", os.path.exists(ZIP_PATH))

Dataset exists: True


In [ ]:
#Extract Dataset.zip
import zipfile
import os

ZIP_PATH = "/content/drive/MyDrive/ML/Dataset.zip"
EXTRACT_PATH = "/content/dataset"

os.makedirs(EXTRACT_PATH, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:
    zip_ref.extractall(EXTRACT_PATH)

print("Dataset extracted successfully!")

Dataset extracted successfully!


In [ ]:
#Check the folder structure
import os

EXTRACT_PATH = "/content/dataset"

for root, dirs, files in os.walk(EXTRACT_PATH):
    level = root.replace(EXTRACT_PATH, "").count(os.sep)

    if level <= 2:
        indent = "    " * level
        print(f"{indent}{os.path.basename(root)}/")

dataset/
    Dataset/
        pattern_present/
        pattern_absent/


In [ ]:
#Count images in each class
import os

DATASET_PATH = "/content/dataset/Dataset"

for class_name in ["pattern_present", "pattern_absent"]:
    class_path = os.path.join(DATASET_PATH, class_name)

    images = [
        f for f in os.listdir(class_path)
        if f.lower().endswith((".jpg", ".jpeg", ".png", ".webp", ".bmp"))
    ]

    print(class_name, ":", len(images), "images")

pattern_present : 1500 images
pattern_absent : 1499 images


In [ ]:
#Check for broken/unreadable image files
from PIL import Image
import os

DATASET_PATH = "/content/dataset/Dataset"

bad_files = []

for class_name in ["pattern_present", "pattern_absent"]:
    class_path = os.path.join(DATASET_PATH, class_name)

    for filename in os.listdir(class_path):
        file_path = os.path.join(class_path, filename)

        try:
            with Image.open(file_path) as img:
                img.verify()
        except Exception:
            bad_files.append(file_path)

print("Broken/unreadable files:", len(bad_files))

Broken/unreadable files: 0


In [ ]:
#Split the dataset into train / validation / test
import os
from sklearn.model_selection import train_test_split
from collections import Counter

DATASET_PATH = "/content/dataset/Dataset"

class_names = ["pattern_absent", "pattern_present"]
image_extensions = (".jpg", ".jpeg", ".png", ".webp", ".bmp")

file_paths = []
labels = []

for label, class_name in enumerate(class_names):
    class_path = os.path.join(DATASET_PATH, class_name)

    for filename in os.listdir(class_path):
        if filename.lower().endswith(image_extensions):
            file_paths.append(os.path.join(class_path, filename))
            labels.append(label)

# First split: 70% train, 30% temporary
train_paths, temp_paths, train_labels, temp_labels = train_test_split(
    file_paths,
    labels,
    test_size=0.30,
    random_state=42,
    stratify=labels
)

# Split temporary set equally: 15% validation, 15% test
val_paths, test_paths, val_labels, test_labels = train_test_split(
    temp_paths,
    temp_labels,
    test_size=0.50,
    random_state=42,
    stratify=temp_labels
)

print("Training images:", len(train_paths))
print("Validation images:", len(val_paths))
print("Test images:", len(test_paths))

print("\nTraining classes:", Counter(train_labels))
print("Validation classes:", Counter(val_labels))
print("Test classes:", Counter(test_labels))

Training images: 2099
Validation images: 450
Test images: 450

Training classes: Counter({1: 1050, 0: 1049})
Validation classes: Counter({0: 225, 1: 225})
Test classes: Counter({1: 225, 0: 225})


In [ ]:
#Create TensorFlow datasets
import tensorflow as tf

IMG_SIZE = 224
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

def load_image(file_path, label):
    image = tf.io.read_file(file_path)

    image = tf.io.decode_image(
        image,
        channels=3,
        expand_animations=False
    )

    image.set_shape([None, None, 3])

    image = tf.image.resize(
        image,
        [IMG_SIZE, IMG_SIZE]
    )

    image = tf.cast(image, tf.float32)

    return image, label


def make_dataset(paths, labels, training=False):
    dataset = tf.data.Dataset.from_tensor_slices((paths, labels))

    if training:
        dataset = dataset.shuffle(
            buffer_size=len(paths),
            seed=42
        )

    dataset = dataset.map(
        load_image,
        num_parallel_calls=AUTOTUNE
    )

    dataset = dataset.batch(BATCH_SIZE)

    dataset = dataset.prefetch(AUTOTUNE)

    return dataset


train_ds = make_dataset(
    train_paths,
    train_labels,
    training=True
)

val_ds = make_dataset(
    val_paths,
    val_labels
)

test_ds = make_dataset(
    test_paths,
    test_labels
)

print("TensorFlow datasets created successfully!")


TensorFlow datasets created successfully!


In [ ]:
# Verify the image batches
for images, labels_batch in train_ds.take(1):
    print("Image batch shape:", images.shape)
    print("Label batch shape:", labels_batch.shape)
    print("Image data type:", images.dtype)
    print("Labels in batch:", labels_batch.numpy())


Image batch shape: (32, 224, 224, 3)
Label batch shape: (32,)
Image data type: <dtype: 'float32'>
Labels in batch: [0 0 1 1 0 0 1 0 0 1 1 0 0 0 1 0 0 1 0 0 0 0 0 1 1 1 1 0 0 1 1 1]


In [ ]:
#Build the pretrained image model
from tensorflow import keras
from tensorflow.keras import layers

# Basic data augmentation
data_augmentation = keras.Sequential([
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.10),
    layers.RandomContrast(0.10),
], name="data_augmentation")


# Load pretrained MobileNetV2
base_model = keras.applications.MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights="imagenet"
)

# Freeze pretrained layers for the first training stage
base_model.trainable = False


# Build our classifier
inputs = keras.Input(shape=(224, 224, 3))

x = data_augmentation(inputs)

# Convert pixels to the format MobileNetV2 expects
x = keras.applications.mobilenet_v2.preprocess_input(x)

x = base_model(x, training=False)

x = layers.GlobalAveragePooling2D()(x)

x = layers.Dropout(0.2)(x)

# Binary output:
# close to 0 = pattern_absent
# close to 1 = pattern_present
outputs = layers.Dense(1, activation="sigmoid")(x)

model = keras.Model(inputs, outputs)

print("Model created successfully!")

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step
Model created successfully!


In [ ]:
#Compile the model
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

print("Model compiled successfully!")

Model compiled successfully!


In [ ]:
#Train the model
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=3,
        restore_best_weights=True
    ),

    keras.callbacks.ModelCheckpoint(
        "/content/drive/MyDrive/ML/best_model.keras",
        monitor="val_loss",
        save_best_only=True
    )
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    callbacks=callbacks
)

Epoch 1/15
66/66 ━━━━━━━━━━━━━━━━━━━━ 28s 250ms/step - accuracy: 0.6427 - loss: 0.6336 - val_accuracy: 0.7422 - val_loss: 0.5217
Epoch 2/15
66/66 ━━━━━━━━━━━━━━━━━━━━ 24s 95ms/step - accuracy: 0.7461 - loss: 0.5255 - val_accuracy: 0.7667 - val_loss: 0.4790
Epoch 3/15
66/66 ━━━━━━━━━━━━━━━━━━━━ 9s 77ms/step - accuracy: 0.7618 - loss: 0.4857 - val_accuracy: 0.7889 - val_loss: 0.4481
Epoch 4/15
66/66 ━━━━━━━━━━━━━━━━━━━━ 5s 77ms/step - accuracy: 0.7785 - loss: 0.4688 - val_accuracy: 0.7711 - val_loss: 0.4616
Epoch 5/15
66/66 ━━━━━━━━━━━━━━━━━━━━ 6s 85ms/step - accuracy: 0.7813 - loss: 0.4533 - val_accuracy: 0.7844 - val_loss: 0.4307
Epoch 6/15
66/66 ━━━━━━━━━━━━━━━━━━━━ 10s 74ms/step - accuracy: 0.7951 - loss: 0.4379 - val_accuracy: 0.7756 - val_loss: 0.4573
Epoch 7/15
66/66 ━━━━━━━━━━━━━━━━━━━━ 5s 78ms/step - accuracy: 0.7947 - loss: 0.4315 - val_accuracy: 0.8022 - val_loss: 0.4171
Epoch 8/15
66/66 ━━━━━━━━━━━━━━━━━━━━ 6s 86ms/step - accuracy: 0.8023 - loss: 0.4221 - val_accuracy: 0.8044

In [ ]:
#Evaluate on the test set
test_loss, test_accuracy = model.evaluate(test_ds)

print("Test loss:", test_loss)
print("Test accuracy:", test_accuracy)
print("Test accuracy %:", test_accuracy * 100)

15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 52ms/step - accuracy: 0.8178 - loss: 0.3818
Test loss: 0.38183915615081787
Test accuracy: 0.8177777528762817
Test accuracy %: 81.77777528762817


In [ ]:
#Create a confusion matrix
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report

# Get true labels
y_true = np.concatenate([
    labels.numpy() for images, labels in test_ds
])

# Get model probabilities
y_prob = model.predict(test_ds).flatten()

# Convert probabilities into 0 / 1 predictions
y_pred = (y_prob >= 0.5).astype(int)

# Confusion matrix
cm = confusion_matrix(y_true, y_pred)

print("Confusion Matrix:")
print(cm)

print("\nClassification Report:")
print(
    classification_report(
        y_true,
        y_pred,
        target_names=["pattern_absent", "pattern_present"]
    )
)

15/15 ━━━━━━━━━━━━━━━━━━━━ 3s 129ms/step
Confusion Matrix:
[[177  48]
 [ 34 191]]

Classification Report:
                 precision    recall  f1-score   support

 pattern_absent       0.84      0.79      0.81       225
pattern_present       0.80      0.85      0.82       225

       accuracy                           0.82       450
      macro avg       0.82      0.82      0.82       450
   weighted avg       0.82      0.82      0.82       450



In [ ]:
#Fine-tune MobileNetV2
# Unfreeze MobileNetV2
base_model.trainable = True

# Keep most layers frozen.
# Only train the last 30 layers.
for layer in base_model.layers[:-30]:
    layer.trainable = False

# Recompile with a MUCH smaller learning rate
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.00001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

print("Fine-tuning setup complete!")

print("Total MobileNetV2 layers:", len(base_model.layers))
print(
    "Trainable MobileNetV2 layers:",
    sum(layer.trainable for layer in base_model.layers)
)

Fine-tuning setup complete!
Total MobileNetV2 layers: 154
Trainable MobileNetV2 layers: 30


In [ ]:
#Fine-tune the model
fine_tune_callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=3,
        restore_best_weights=True
    ),

    keras.callbacks.ModelCheckpoint(
        "/content/drive/MyDrive/ML/best_finetuned_model.keras",
        monitor="val_loss",
        save_best_only=True
    )
]

fine_tune_history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=fine_tune_callbacks
)


Epoch 1/10
66/66 ━━━━━━━━━━━━━━━━━━━━ 23s 161ms/step - accuracy: 0.6722 - loss: 0.6576 - val_accuracy: 0.7978 - val_loss: 0.4393
Epoch 2/10
66/66 ━━━━━━━━━━━━━━━━━━━━ 16s 112ms/step - accuracy: 0.7851 - loss: 0.4406 - val_accuracy: 0.8156 - val_loss: 0.4198
Epoch 3/10
66/66 ━━━━━━━━━━━━━━━━━━━━ 7s 100ms/step - accuracy: 0.8056 - loss: 0.4100 - val_accuracy: 0.8378 - val_loss: 0.3832
Epoch 4/10
66/66 ━━━━━━━━━━━━━━━━━━━━ 7s 106ms/step - accuracy: 0.8390 - loss: 0.3585 - val_accuracy: 0.8333 - val_loss: 0.3583
Epoch 5/10
66/66 ━━━━━━━━━━━━━━━━━━━━ 7s 101ms/step - accuracy: 0.8471 - loss: 0.3480 - val_accuracy: 0.8400 - val_loss: 0.3386
Epoch 6/10
66/66 ━━━━━━━━━━━━━━━━━━━━ 8s 116ms/step - accuracy: 0.8576 - loss: 0.3207 - val_accuracy: 0.8333 - val_loss: 0.3367
Epoch 7/10
66/66 ━━━━━━━━━━━━━━━━━━━━ 9s 104ms/step - accuracy: 0.8742 - loss: 0.3000 - val_accuracy: 0.8422 - val_loss: 0.3235
Epoch 8/10
66/66 ━━━━━━━━━━━━━━━━━━━━ 7s 111ms/step - accuracy: 0.8690 - loss: 0.2933 - val_accuracy: 

In [ ]:
#Test the fine-tuned model
test_loss_ft, test_accuracy_ft = model.evaluate(test_ds)

print("Fine-tuned test loss:", test_loss_ft)
print("Fine-tuned test accuracy:", test_accuracy_ft)
print("Fine-tuned test accuracy %:", test_accuracy_ft * 100)

15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 51ms/step - accuracy: 0.9000 - loss: 0.2818
Fine-tuned test loss: 0.2817886173725128
Fine-tuned test accuracy: 0.8999999761581421
Fine-tuned test accuracy %: 89.99999761581421


In [ ]:
#Check the new confusion matrix
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report

# True labels
y_true = np.concatenate([
    labels.numpy() for images, labels in test_ds
])

# Fine-tuned model predictions
y_prob = model.predict(test_ds).flatten()

# Probability >= 0.5 means pattern_present
y_pred = (y_prob >= 0.5).astype(int)

# Confusion matrix
cm = confusion_matrix(y_true, y_pred)

print("Fine-tuned Confusion Matrix:")
print(cm)

print("\nClassification Report:")
print(
    classification_report(
        y_true,
        y_pred,
        target_names=["pattern_absent", "pattern_present"]
    )
)


15/15 ━━━━━━━━━━━━━━━━━━━━ 3s 127ms/step
Fine-tuned Confusion Matrix:
[[207  18]
 [ 27 198]]

Classification Report:
                 precision    recall  f1-score   support

 pattern_absent       0.88      0.92      0.90       225
pattern_present       0.92      0.88      0.90       225

       accuracy                           0.90       450
      macro avg       0.90      0.90      0.90       450
   weighted avg       0.90      0.90      0.90       450



In [ ]:
#Save your finished model
FINAL_MODEL_PATH = "/content/drive/MyDrive/ML/pattern_classifier_final.keras"

model.save(FINAL_MODEL_PATH)

print("Final model saved successfully!")
print("Location:", FINAL_MODEL_PATH)
print()
print("Class mapping:")
print("0 = pattern_absent")
print("1 = pattern_present")

Final model saved successfully!
Location: /content/drive/MyDrive/ML/pattern_classifier_final.keras

Class mapping:
0 = pattern_absent
1 = pattern_present


In [ ]:
NEW_IMAGE_PATH = "/content/drive/MyDrive/ML/test_images/test1.jpg"

import os

print("Image exists:", os.path.exists(NEW_IMAGE_PATH))
print("Path:", NEW_IMAGE_PATH)

Image exists: True
Path: /content/drive/MyDrive/ML/test_images/test1.jpg


In [ ]:
#Classify the new image

import tensorflow as tf
import numpy as np

# Load the image
img = tf.keras.utils.load_img(
    NEW_IMAGE_PATH,
    target_size=(224, 224)
)

# Convert image to array
img_array = tf.keras.utils.img_to_array(img)

# Add batch dimension: (224,224,3) -> (1,224,224,3)
img_array = tf.expand_dims(img_array, axis=0)

# Predict
probability = model.predict(img_array, verbose=0)[0][0]

# Convert prediction to readable result
if probability >= 0.5:
    prediction = "pattern_present"
    confidence = probability
else:
    prediction = "pattern_absent"
    confidence = 1 - probability

print("Prediction:", prediction)
print("Confidence:", f"{confidence * 100:.2f}%")
print("Raw model score:", f"{probability:.4f}")

Prediction: pattern_present
Confidence: 99.77%
Raw model score: 0.9977


In [ ]:
#Test an image you know is pattern_absent
NEW_IMAGE_PATH = "/content/drive/MyDrive/ML/test_images/test2.jpg"

import os

print("Image exists:", os.path.exists(NEW_IMAGE_PATH))

Image exists: True


In [ ]:
import tensorflow as tf

img = tf.keras.utils.load_img(
    NEW_IMAGE_PATH,
    target_size=(224, 224)
)

img_array = tf.keras.utils.img_to_array(img)
img_array = tf.expand_dims(img_array, axis=0)

probability = model.predict(img_array, verbose=0)[0][0]

if probability >= 0.5:
    prediction = "pattern_present"
    confidence = probability
else:
    prediction = "pattern_absent"
    confidence = 1 - probability

print("Prediction:", prediction)
print("Confidence:", f"{confidence * 100:.2f}%")
print("Raw model score:", f"{probability:.4f}")

Prediction: pattern_absent
Confidence: 83.49%
Raw model score: 0.1651
